In [ ]:
# =============================================
# ES2 — Definizioni e ricerca onomasiologica
# =============================================

!pip -q install gensim nltk sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 44.8 MB/s eta 0:00:00


In [ ]:
import re
import math
import numpy as np
import pandas as pd
from itertools import combinations
from typing import Dict, List, Tuple, Optional

import nltk
from nltk.corpus import stopwords
from nltk.stem.snowball import SnowballStemmer

from sentence_transformers import SentenceTransformer

# -----------------------------
# 0) Configurazione risorse NLTK
# -----------------------------
def ensure_nltk():
    for res in ["stopwords"]:
        try:
            nltk.data.find(f"corpora/{res}")
        except LookupError:
            nltk.download(res)

ensure_nltk()

ITALIAN_STOPWORDS = set(stopwords.words("italian"))
STEMMER = SnowballStemmer("italian")

# -----------------------------
# 1) Caricamento e analisi dataset
# -----------------------------
DATA_PATH = "dataset_definizioni_TLN_25.xlsx"
df = pd.read_excel(DATA_PATH)

# normalizza nomi colonne
df.columns = [str(c).strip() for c in df.columns]

# definizione dei codici di categoria previsti dal dataset:
CODES = {"CG", "CS", "AG", "AS"}

# 1. Identifica la colonna che contiene i codici categoria
def find_category_column(df: pd.DataFrame) -> str:
    for col in df.columns:
        vals = df[col].dropna().astype(str).str.strip().unique().tolist()
        # se almeno metà dei valori non-null sta in CODES -> colonna categoria
        if len(vals) > 0:
            hit = sum(v in CODES for v in vals)
            if hit >= max(1, int(0.5 * len(vals))):
                return col
    raise ValueError("Non trovo la colonna categoria (CG/CS/AG/AS). Controlla il file.")

cat_col = find_category_column(df)

# 2. Identifica la colonna contentente il termine da definire
def find_term_column(df: pd.DataFrame) -> str:
    for cand in ["Termine", "termine", "TERMINE"]:
        if cand in df.columns:
            return cand
    # fallback: seconda colonna se prima è categoria
    cols = list(df.columns)
    if cols[0] == cat_col and len(cols) >= 2:
        return cols[1]
    raise ValueError("Non trovo la colonna 'Termine'. Controlla il file.")

term_col = find_term_column(df)

# 3. Identifica le colonne che contengono le definizioni dei termini
def_cols = [c for c in df.columns if c not in {cat_col, term_col}]
def_cols = [c for c in def_cols if bool(re.match(r"^P\d+$", c.strip(), re.IGNORECASE))]

if len(def_cols) == 0:
    raise ValueError("Non trovo colonne definizione (P1, P2, ...). Controlla il file.")

# -----------------------------
# 2) Preprocessing (token -> stopword removal + stemming)
# -----------------------------
TOKEN_RE = re.compile(r"[a-zàèéìòù]+", re.IGNORECASE)

def preprocess(text: str) -> List[str]:
    if not isinstance(text, str) or not text.strip():
        return []
    text = text.lower()
    tokens = TOKEN_RE.findall(text)
    out = []
    for t in tokens:
        if t in ITALIAN_STOPWORDS:
            continue
        if len(t) < 2:
            continue
        out.append(STEMMER.stem(t))
    return out

# -----------------------------
# 3) Similarità Lessicale: Jaccard su insiemi di token
# -----------------------------
def simlex_jaccard(toks_a: List[str], toks_b: List[str]) -> float:
    A, B = set(toks_a), set(toks_b)
    if not A and not B:
        return 0.0
    return len(A & B) / len(A | B)

# -----------------------------
# 4) Similarità Semantica: Sentence Embeddings (pretrained) + cosine
# -----------------------------

# modello multilingue adatto anche all'italiano (frasi -> vettori)
SEM_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
sem_model = SentenceTransformer(SEM_MODEL_NAME)

# similarità coseno tra due vettori
def cosine(a: np.ndarray, b: np.ndarray) -> float:
    na = np.linalg.norm(a)
    nb = np.linalg.norm(b)
    if na == 0 or nb == 0:
        return 0.0
    return float(np.dot(a, b) / (na * nb))

# trasforma una lista di definizioni in una matrice di vettori (embeddings)
def def_embeddings(defs: List[str]) -> np.ndarray:
    return sem_model.encode(defs, normalize_embeddings=True, show_progress_bar=False)

# -----------------------------
# 5) Calcolo della similarità pairwise per ogni termine
# -----------------------------
def get_definitions_for_row(row: pd.Series) -> List[str]:
    defs = []
    for c in def_cols:
        v = row.get(c, None)
        if isinstance(v, str) and v.strip():
            defs.append(v.strip())
    return defs

records = [] # memorizzerà le medie di similarità per ogni termine

for _, row in df.iterrows():
    cat = str(row[cat_col]).strip()
    term = str(row[term_col]).strip()

    defs = get_definitions_for_row(row)
    if len(defs) < 2:
        # con meno di 2 definizioni non ha senso la similarità pairwise
        continue

    # preprocess per simlex
    toks_list = [preprocess(d) for d in defs]
    # embeddings per ciascuna definizione per simsem
    embs = def_embeddings(defs)

    simlex_vals = []
    simsem_vals = []

    # confronto incrociato tra tutte le definizioni di un termine
    for i, j in combinations(range(len(defs)), 2):
        # calcolo jaccard (simlex)
        simlex_vals.append(simlex_jaccard(toks_list[i], toks_list[j]))
        # calcolo coseno (simsem)
        simsem_vals.append(cosine(embs[i], embs[j]))

    # salvataggio della media delle similarità per il termine corrente
    records.append({
        "Categoria": cat,
        "Termine": term,
        "n_def": len(defs),
        "n_pairs": len(simlex_vals),
        "simlex_mean_term": float(np.mean(simlex_vals)) if simlex_vals else np.nan,
        "simsem_mean_term": float(np.mean(simsem_vals)) if simsem_vals else np.nan,
    })

# creazione DataFrame con i risultati per ogni singolo termine
term_stats = pd.DataFrame(records)

# -----------------------------
# 6) Aggregazione per categoria (CG/CS/AG/AS)
# -----------------------------

# raggruppa i risultati per categoria e calcola la media generale
agg = (
    term_stats
    .groupby("Categoria", as_index=False)
    .agg(
        Termine=("Termine", "first"),
        simlex_mean=("simlex_mean_term", "mean"),
        simsem_mean=("simsem_mean_term", "mean"),
    )
)

print("\n=== Aggregazione per categoria ===")
print(agg)


=== Aggregazione per categoria ===
  Categoria      Termine  simlex_mean  simsem_mean
0        AG     Pericolo     0.112228     0.652930
1        AS    Euristica     0.041862     0.454166
2        CG    Pantalone     0.186461     0.659362
3        CS  Microscopio     0.154150     0.632523
